In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pycaret.classification import *
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("C:\\Users\\ripa_\\Desktop\\Programing\\IndyCar_Project\\LigaMX\\datasets\\LigaMX_dataset_v5.csv")

In [3]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Date", "Equipo"])

In [4]:
df.head()

,Date,Time,Round,Day,Venue,Opponent,Referee,Equipo,Torneo,Temporada,...,TeamAwayForm5,OpponentForm5,OpponentWinRate5,OpponentSeasonPts,OpponentHomeForm5,OpponentAwayForm5,H2HWinRate,H2HLast5,FormDiff,SeasonPtsDiff
0,2014-07-18,19:30,Jornada 1,Fri,Away,Tijuana,Luis Enrique Santander,Puebla,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0
1,2014-07-18,19:30,Jornada 1,Fri,Away,Queretaro,Erick Yair Miranda,Pumas UNAM,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0
2,2014-07-18,19:30,Jornada 1,Fri,Home,Pumas UNAM,Erick Yair Miranda,Queretaro,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0
3,2014-07-18,19:30,Jornada 1,Fri,Home,Puebla,Luis Enrique Santander,Tijuana,Apertura,2014,...,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0
4,2014-07-19,21:00,Jornada 1,Sat,Home,Tigres UANL,Erim Ramírez,Atlas,Apertura,2014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
unique_matches = df.drop_duplicates("match_key")[["match_key", "Date"]].sort_values("Date")
split = int(len(unique_matches)*0.95)

train_matches = unique_matches.iloc[:split]["match_key"]
unseen_matches = unique_matches.iloc[split:]["match_key"]

train_df = df[df["match_key"].isin(train_matches)]
unseen_df = df[df["match_key"].isin(unseen_matches)]

drop_cols = [
   "Date", "Time", "Round", "Day", "Venue", "Opponent", "Referee",
    "Equipo", "Torneo", "Temporada", "Result", "Points", "match_key"
]


data = train_df.drop(columns=drop_cols)
data_unseen = unseen_df.drop(columns=drop_cols)

print(data.corr(numeric_only=True)["ResultID"].sort_values())

OponentElo          -0.111467
OpponentAwayForm5   -0.028216
DayID               -0.026048
OpponentSeasonPts   -0.024546
OpponentForm5       -0.018711
OpponentWinRate5    -0.018032
TemporadaID         -0.014317
TorneoID            -0.011905
RoundID             -0.011677
TeamID              -0.003892
OpponentHomeForm5    0.001319
RefereeID            0.008926
TimeID               0.014034
TeamSeasonPts        0.023638
OpponentID           0.026167
TeamHomeForm5        0.026964
TeamWinRate5         0.051696
H2HWinRate           0.055712
TeamAwayForm5        0.057775
TeamForm5            0.060961
FormDiff             0.061274
SeasonPtsDiff        0.076303
VenueID              0.109693
H2HLast5             0.125813
TeamElo              0.145310
EloDiff              0.184746
ResultID             1.000000
Name: ResultID, dtype: float64


In [6]:
df = df.drop(columns=drop_cols)

In [7]:
print(df.columns.tolist())

['RoundID', 'VenueID', 'OpponentID', 'TeamID', 'RefereeID', 'TemporadaID', 'TorneoID', 'TimeID', 'DayID', 'ResultID', 'TeamElo', 'OponentElo', 'EloDiff', 'TeamForm5', 'TeamWinRate5', 'TeamSeasonPts', 'TeamHomeForm5', 'TeamAwayForm5', 'OpponentForm5', 'OpponentWinRate5', 'OpponentSeasonPts', 'OpponentHomeForm5', 'OpponentAwayForm5', 'H2HWinRate', 'H2HLast5', 'FormDiff', 'SeasonPtsDiff']


In [8]:
exp = setup(
    data=data, 
    target="ResultID", 
    session_id=123, 
    fold_strategy="timeseries",
    data_split_shuffle=False,
    data_split_stratify=False,
    fold_shuffle=False
)

,Description,Value
0,Session id,123
1,Target,ResultID
2,Target type,Multiclass
3,Original data shape,"(7164, 27)"
4,Transformed data shape,"(7164, 27)"
5,Transformed train set shape,"(5014, 27)"
6,Transformed test set shape,"(2150, 27)"
7,Numeric features,26
8,Rows with missing values,5.4%
9,Preprocess,True


In [9]:
compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
ridge,Ridge Classifier,0.4989,0.0000,0.4989,0.4475,0.4369,0.2195,0.2388,0.0120
lr,Logistic Regression,0.4971,0.0000,0.4971,0.4635,0.4514,0.2177,0.2353,0.9930
lda,Linear Discriminant Analysis,0.4936,0.0000,0.4936,0.4534,0.4424,0.2155,0.2318,0.0100
rf,Random Forest Classifier,0.4916,0.6618,0.4916,0.4740,0.4642,0.2148,0.2223,0.1350
et,Extra Trees Classifier,0.4892,0.6621,0.4892,0.4775,0.4649,0.2117,0.2187,0.1020
gbc,Gradient Boosting Classifier,0.4890,0.0000,0.4890,0.4761,0.4677,0.2140,0.2201,0.5060
lightgbm,Light Gradient Boosting Machine,0.4884,0.6674,0.4884,0.4770,0.4731,0.2136,0.2177,0.3930
ada,Ada Boost Classifier,0.4866,0.0000,0.4866,0.4810,0.4700,0.2132,0.2191,0.0600
catboost,CatBoost Classifier,0.4846,0.6667,0.4846,0.4757,0.4712,0.2095,0.2134,4.0500
nb,Naive Bayes,0.4736,0.6554,0.4736,0.4645,0.4550,0.1932,0.1989,0.0110


RidgeClassifier(alpha=1.0, class_weight=None, copy_X=True, fit_intercept=True,
                max_iter=None, positive=False, random_state=123, solver='auto',
                tol=0.0001)

In [10]:
cat = create_model('catboost')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4637,0.6208,0.4637,0.4677,0.4625,0.1697,0.1709
1,0.4681,0.6561,0.4681,0.4750,0.4380,0.2011,0.2155
2,0.4505,0.6511,0.4505,0.4563,0.4531,0.1739,0.1740
3,0.4769,0.6487,0.4769,0.4647,0.4681,0.2010,0.2022
4,0.4857,0.6766,0.4857,0.4727,0.4764,0.2118,0.2132
5,0.4681,0.6570,0.4681,0.4601,0.4627,0.1877,0.1883
6,0.5363,0.7082,0.5363,0.5130,0.5152,0.2729,0.2779
7,0.5451,0.7336,0.5451,0.5257,0.5280,0.2891,0.2930
8,0.5099,0.6877,0.5099,0.4936,0.4834,0.2377,0.2454


In [11]:
cat_tune = tune_model(cat)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4769,0.6267,0.4769,0.4779,0.4699,0.1876,0.1910
1,0.4308,0.6703,0.4308,0.4257,0.3698,0.1445,0.1637
2,0.4967,0.6765,0.4967,0.4920,0.4937,0.2387,0.2391
3,0.4813,0.6711,0.4813,0.4723,0.4750,0.2092,0.2100
4,0.5143,0.7076,0.5143,0.4909,0.4933,0.2498,0.2545
5,0.5011,0.6778,0.5011,0.4687,0.4733,0.2270,0.2329
6,0.5407,0.7103,0.5407,0.5128,0.5186,0.2804,0.2851
7,0.5714,0.7458,0.5714,0.5518,0.5387,0.3219,0.3329
8,0.5231,0.6944,0.5231,0.4902,0.4731,0.2518,0.2669


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [23]:
predict_model(cat_tune);
#predict_model(cat);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,CatBoost Classifier,0.5391,0.7028,0.5391,0.5039,0.5038,0.2798,0.2895


In [25]:
newpred = predict_model(cat_tune, data=data_unseen)
#newpred = predict_model(cat, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,CatBoost Classifier,0.5688,0.7034,0.5688,0.5422,0.5454,0.2964,0.3038


In [12]:
rf = create_model('rf')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4703,0.6239,0.4703,0.4655,0.4565,0.1699,0.1748
1,0.4396,0.6184,0.4396,0.4604,0.3837,0.1576,0.1773
2,0.4857,0.6456,0.4857,0.4676,0.4732,0.2173,0.2190
3,0.4791,0.6497,0.4791,0.4729,0.4699,0.2016,0.2042
4,0.4967,0.6819,0.4967,0.4732,0.4769,0.2231,0.2271
5,0.4989,0.6679,0.4989,0.4643,0.4678,0.2220,0.2289
6,0.4945,0.6721,0.4945,0.4697,0.4780,0.2123,0.2144
7,0.5604,0.7152,0.5604,0.5340,0.5261,0.3045,0.3154
8,0.5187,0.6900,0.5187,0.4841,0.4801,0.2486,0.2591


In [13]:
rf_tune = tune_model(rf)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4462,0.6244,0.4462,0.4644,0.4256,0.1532,0.1630
1,0.4593,0.6523,0.4593,0.4528,0.4204,0.1879,0.2038
2,0.5033,0.6741,0.5033,0.5137,0.5056,0.2543,0.2556
3,0.4747,0.6504,0.4747,0.4845,0.4765,0.2089,0.2102
4,0.4769,0.6989,0.4769,0.4834,0.4793,0.2105,0.2109
5,0.4989,0.6761,0.4989,0.4917,0.4938,0.2360,0.2368
6,0.4945,0.6789,0.4945,0.4869,0.4899,0.2228,0.2232
7,0.5538,0.7196,0.5538,0.5437,0.5478,0.3122,0.3128
8,0.5187,0.6848,0.5187,0.5074,0.5078,0.2599,0.2626


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [28]:
predict_model(rf_tune);
#predict_model(rf);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.5293,0.6804,0.5293,0.4941,0.4955,0.2650,0.2737


In [29]:
newpred = predict_model(rf_tune, data=data_unseen)
#newpred = predict_model(rf, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.5507,0.6799,0.5507,0.5130,0.5182,0.2596,0.2695


In [14]:
lgbm = create_model('lightgbm')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4659,0.6104,0.4659,0.4657,0.4588,0.1639,0.1668
1,0.4505,0.6327,0.4505,0.4513,0.4224,0.1747,0.1857
2,0.4857,0.6576,0.4857,0.4832,0.4844,0.2231,0.2231
3,0.4681,0.6449,0.4681,0.4561,0.4601,0.1884,0.1893
4,0.4725,0.6758,0.4725,0.4631,0.4668,0.1949,0.1954
5,0.4615,0.6638,0.4615,0.4439,0.4453,0.1694,0.1722
6,0.5407,0.7044,0.5407,0.5186,0.5220,0.2815,0.2856
7,0.5516,0.7188,0.5516,0.5300,0.5363,0.3020,0.3047
8,0.5341,0.7058,0.5341,0.5187,0.5029,0.2737,0.2840


In [15]:
lgbm_tune = tune_model(lgbm)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4857,0.6257,0.4857,0.4785,0.4555,0.1728,0.1865
1,0.4462,0.6658,0.4462,0.3944,0.3652,0.1672,0.1904
2,0.5275,0.6667,0.5275,0.5138,0.5084,0.2763,0.2823
3,0.4791,0.6614,0.4791,0.4717,0.4617,0.1966,0.2014
4,0.5033,0.6984,0.5033,0.4653,0.4673,0.2269,0.2355
5,0.5209,0.6830,0.5209,0.5222,0.4708,0.2481,0.2650
6,0.5560,0.6988,0.5560,0.5594,0.5039,0.2898,0.3079
7,0.5714,0.7344,0.5714,0.5842,0.5266,0.3160,0.3335
8,0.5341,0.7038,0.5341,0.5011,0.4675,0.2652,0.2870


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [31]:
predict_model(lgbm_tune);
#predict_model(lgbm);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.5433,0.6958,0.5433,0.5096,0.4698,0.2755,0.2998


[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


In [33]:
#newpred = predict_model(lgbm_tune, data=data_unseen)
newpred = predict_model(lgbm, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.5326,0.6975,0.5326,0.5259,0.5271,0.2537,0.2549


In [16]:
et = create_model('et')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4352,0.6213,0.4352,0.4379,0.4284,0.1228,0.1253
1,0.4176,0.6402,0.4176,0.4165,0.3722,0.1250,0.1399
2,0.4703,0.6213,0.4703,0.4658,0.4675,0.1992,0.1995
3,0.4857,0.6463,0.4857,0.4749,0.4701,0.2083,0.2126
4,0.5077,0.6827,0.5077,0.4850,0.4908,0.2422,0.2452
5,0.5143,0.6710,0.5143,0.4994,0.4971,0.2500,0.2545
6,0.5253,0.6954,0.5253,0.4880,0.4912,0.2496,0.2576
7,0.5692,0.7125,0.5692,0.5536,0.5450,0.3217,0.3297
8,0.4923,0.6858,0.4923,0.4729,0.4530,0.2052,0.2158


In [17]:
et_tune = tune_model(et)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4725,0.6393,0.4725,0.4681,0.4650,0.1785,0.1808
1,0.4330,0.6498,0.4330,0.4769,0.3603,0.1476,0.1732
2,0.4637,0.6507,0.4637,0.4446,0.4500,0.1826,0.1844
3,0.4857,0.6566,0.4857,0.4975,0.4332,0.1956,0.2110
4,0.5209,0.7089,0.5209,0.4263,0.4541,0.2473,0.2654
5,0.5143,0.6740,0.5143,0.5106,0.4390,0.2332,0.2561
6,0.5582,0.6953,0.5582,0.5614,0.4945,0.2909,0.3125
7,0.5758,0.7264,0.5758,0.6754,0.5114,0.3184,0.3432
8,0.5275,0.6960,0.5275,0.4949,0.4549,0.2535,0.2765


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [35]:
predict_model(et_tune);
#predict_model(et);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extra Trees Classifier,0.5326,0.6928,0.5326,0.3953,0.4536,0.2573,0.2823


In [37]:
#newpred = predict_model(et_tune, data=data_unseen)
newpred = predict_model(et, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extra Trees Classifier,0.5435,0.6771,0.5435,0.5109,0.5142,0.2586,0.2663


In [18]:
xgb = create_model('xgboost')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4308,0.5983,0.4308,0.4158,0.4222,0.1018,0.1022
1,0.4330,0.6308,0.4330,0.4266,0.4110,0.1484,0.1550
2,0.4418,0.6249,0.4418,0.4425,0.4420,0.1581,0.1582
3,0.4549,0.6348,0.4549,0.4420,0.4453,0.1668,0.1680
4,0.4681,0.6562,0.4681,0.4603,0.4635,0.1893,0.1896
5,0.4857,0.6638,0.4857,0.4700,0.4725,0.2089,0.2112
6,0.5077,0.6919,0.5077,0.4802,0.4896,0.2324,0.2350
7,0.5099,0.6969,0.5099,0.5031,0.5061,0.2468,0.2471
8,0.5297,0.6978,0.5297,0.5142,0.5041,0.2689,0.2770


In [19]:
xgb_tune = tune_model(xgb)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4813,0.6400,0.4813,0.4156,0.4095,0.1370,0.1725
1,0.4220,0.6620,0.4220,0.3052,0.3318,0.1299,0.1705
2,0.4879,0.6635,0.4879,0.3588,0.4058,0.2021,0.2339
3,0.4659,0.6611,0.4659,0.3448,0.3909,0.1610,0.1831
4,0.5297,0.6842,0.5297,0.3966,0.4490,0.2569,0.2890
5,0.5253,0.6711,0.5253,0.3885,0.4443,0.2500,0.2785
6,0.5582,0.7011,0.5582,0.4247,0.4809,0.2886,0.3153
7,0.5560,0.7375,0.5560,0.4243,0.4797,0.2848,0.3114
8,0.5253,0.7003,0.5253,0.3875,0.4452,0.2488,0.2749


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [39]:
#predict_model(xgb_tune);
predict_model(xgb);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extreme Gradient Boosting,0.5149,0.6752,0.5149,0.4948,0.5008,0.2521,0.2544


In [41]:
#newpred = predict_model(xgb_tune, data=data_unseen)
newpred = predict_model(xgb, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extreme Gradient Boosting,0.5109,0.6658,0.5109,0.5017,0.5056,0.2245,0.2249


In [20]:
nb = create_model('nb')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4549,0.6280,0.4549,0.4574,0.4420,0.1520,0.1574
1,0.4505,0.6365,0.4505,0.4552,0.3965,0.1743,0.1963
2,0.4835,0.6589,0.4835,0.4984,0.4839,0.2255,0.2284
3,0.4352,0.6227,0.4352,0.4477,0.4363,0.1501,0.1516
4,0.4813,0.6799,0.4813,0.4717,0.4742,0.2083,0.2095
5,0.4923,0.6733,0.4923,0.4739,0.4770,0.2185,0.2215
6,0.4681,0.6567,0.4681,0.4546,0.4602,0.1796,0.1801
7,0.5363,0.7077,0.5363,0.5043,0.5068,0.2691,0.2762
8,0.4879,0.6665,0.4879,0.4553,0.4534,0.2014,0.2093


In [21]:
nb_tune = tune_model(nb)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4659,0.5846,0.4659,0.4432,0.4422,0.1331,0.1365
1,0.4132,0.6154,0.4132,0.2771,0.3313,0.1172,0.1354
2,0.4901,0.6677,0.4901,0.3512,0.4091,0.2055,0.2291
3,0.4725,0.6256,0.4725,0.3438,0.3979,0.1710,0.1893
4,0.4725,0.6508,0.4725,0.3469,0.4001,0.1667,0.1837
5,0.5319,0.6670,0.5319,0.3904,0.4501,0.2604,0.2872
6,0.4879,0.6205,0.4879,0.3699,0.4208,0.1752,0.1902
7,0.5341,0.6821,0.5341,0.4049,0.4606,0.2495,0.2709
8,0.4791,0.6435,0.4791,0.3528,0.4063,0.1756,0.1933


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [43]:
#predict_model(nb_tune);
predict_model(nb);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Naive Bayes,0.5140,0.6752,0.5140,0.4818,0.4856,0.2432,0.2496


In [46]:
#newpred = predict_model(nb_tune, data=data_unseen)
newpred = predict_model(nb, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Naive Bayes,0.5181,0.6209,0.5181,0.4846,0.4920,0.2098,0.2157


In [54]:
blend1 = blend_models([cat_tune, lgbm])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4791,0.6207,0.4791,0.4777,0.4708,0.1846,0.1881
1,0.4484,0.6533,0.4484,0.4495,0.4106,0.1713,0.1866
2,0.4989,0.6701,0.4989,0.4955,0.4970,0.2426,0.2427
3,0.4813,0.6608,0.4813,0.4696,0.4733,0.2085,0.2095
4,0.5011,0.6940,0.5011,0.4845,0.4890,0.2341,0.2361
5,0.4791,0.6747,0.4791,0.4557,0.4573,0.1943,0.1986
6,0.5538,0.7137,0.5538,0.5269,0.5229,0.2954,0.3044
7,0.5538,0.7368,0.5538,0.5269,0.5314,0.3008,0.3059
8,0.5385,0.7086,0.5385,0.5225,0.4947,0.2769,0.2920


In [55]:
blend1_tune = tune_model(blend1)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4725,0.6251,0.4725,0.4730,0.4658,0.1775,0.1806
1,0.4637,0.6659,0.4637,0.4744,0.4143,0.1942,0.2161
2,0.4835,0.6759,0.4835,0.4756,0.4789,0.2178,0.2182
3,0.4857,0.6690,0.4857,0.4747,0.4783,0.2157,0.2166
4,0.5209,0.7030,0.5209,0.5023,0.5066,0.2635,0.2662
5,0.4989,0.6778,0.4989,0.4682,0.4704,0.2228,0.2292
6,0.5407,0.7142,0.5407,0.5086,0.5105,0.2757,0.2832
7,0.5714,0.7436,0.5714,0.5453,0.5419,0.3242,0.3330
8,0.5363,0.7033,0.5363,0.5037,0.4844,0.2722,0.2888


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [58]:
predict_model(blend1_tune);
#predict_model(blend1);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5330,0.7042,0.5330,0.4970,0.4984,0.2706,0.2797


In [59]:
newpred = predict_model(blend1_tune, data=data_unseen)
#newpred = predict_model(blend1, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5797,0.7079,0.5797,0.5600,0.5583,0.3116,0.3202


In [60]:
blend2 = blend_models([cat_tune, et])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4637,0.6296,0.4637,0.4674,0.4547,0.1694,0.1738
1,0.4505,0.6693,0.4505,0.4597,0.3911,0.1743,0.1995
2,0.4857,0.6580,0.4857,0.4798,0.4820,0.2219,0.2223
3,0.4857,0.6666,0.4857,0.4749,0.4769,0.2137,0.2153
4,0.5341,0.7048,0.5341,0.5073,0.5106,0.2798,0.2856
5,0.5319,0.6803,0.5319,0.5111,0.5027,0.2727,0.2816
6,0.5495,0.7112,0.5495,0.5116,0.5148,0.2882,0.2971
7,0.5868,0.7393,0.5868,0.5675,0.5543,0.3465,0.3581
8,0.5341,0.7001,0.5341,0.5100,0.4817,0.2682,0.2854


In [61]:
blend2_tune = tune_model(blend2)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4637,0.6290,0.4637,0.4656,0.4532,0.1684,0.1730
1,0.4615,0.6675,0.4615,0.4771,0.4027,0.1909,0.2196
2,0.4747,0.6538,0.4747,0.4699,0.4716,0.2059,0.2062
3,0.4835,0.6647,0.4835,0.4731,0.4746,0.2101,0.2117
4,0.5341,0.7034,0.5341,0.5060,0.5113,0.2806,0.2858
5,0.5385,0.6801,0.5385,0.5241,0.5111,0.2829,0.2921
6,0.5516,0.7100,0.5516,0.5155,0.5161,0.2910,0.3006
7,0.5824,0.7371,0.5824,0.5663,0.5519,0.3399,0.3508
8,0.5297,0.6986,0.5297,0.5066,0.4779,0.2613,0.2780


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [79]:
#predict_model(blend2_tune);
predict_model(blend2);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5349,0.7014,0.5349,0.4910,0.4894,0.2699,0.2829


In [80]:
#newpred = predict_model(blend2_tune, data=data_unseen)
newpred = predict_model(blend2, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5688,0.6953,0.5688,0.5322,0.5338,0.2921,0.3036


In [62]:
blend3 = blend_models([cat_tune, nb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4637,0.6285,0.4637,0.4689,0.4509,0.1668,0.1730
1,0.4505,0.6558,0.4505,0.4480,0.3875,0.1742,0.1976
2,0.4923,0.6734,0.4923,0.4994,0.4923,0.2355,0.2371
3,0.4637,0.6469,0.4637,0.4654,0.4627,0.1885,0.1892
4,0.4967,0.6978,0.4967,0.4792,0.4835,0.2274,0.2297
5,0.5033,0.6807,0.5033,0.4793,0.4836,0.2336,0.2375
6,0.4967,0.6855,0.4967,0.4708,0.4805,0.2168,0.2186
7,0.5626,0.7346,0.5626,0.5487,0.5285,0.3066,0.3186
8,0.5099,0.6875,0.5099,0.4908,0.4702,0.2330,0.2447


In [63]:
blend3_tune = tune_model(blend3)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4791,0.6298,0.4791,0.4846,0.4667,0.1886,0.1954
1,0.4418,0.6656,0.4418,0.4302,0.3755,0.1610,0.1834
2,0.5121,0.6770,0.5121,0.5109,0.5096,0.2626,0.2636
3,0.4879,0.6624,0.4879,0.4836,0.4845,0.2216,0.2222
4,0.5165,0.7055,0.5165,0.4990,0.5012,0.2557,0.2591
5,0.5143,0.6807,0.5143,0.4868,0.4894,0.2482,0.2539
6,0.5275,0.7015,0.5275,0.5022,0.5094,0.2626,0.2657
7,0.5758,0.7443,0.5758,0.5711,0.5412,0.3269,0.3406
8,0.5253,0.6946,0.5253,0.4988,0.4820,0.2568,0.2701


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [82]:
predict_model(blend3_tune);
#predict_model(blend3);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5367,0.7021,0.5367,0.4935,0.4958,0.2746,0.2859


In [84]:
newpred = predict_model(blend3_tune, data=data_unseen)
#newpred = predict_model(blend3, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5507,0.6827,0.5507,0.5127,0.5165,0.2587,0.2695


In [64]:
blend4 = blend_models([cat_tune, et, nb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4813,0.6306,0.4813,0.4897,0.4688,0.1926,0.2001
1,0.4593,0.6571,0.4593,0.4638,0.3994,0.1875,0.2121
2,0.4945,0.6663,0.4945,0.4994,0.4929,0.2386,0.2405
3,0.4703,0.6535,0.4703,0.4675,0.4673,0.1958,0.1965
4,0.5077,0.7004,0.5077,0.4898,0.4921,0.2422,0.2456
5,0.5231,0.6826,0.5231,0.4941,0.4955,0.2608,0.2678
6,0.5099,0.6932,0.5099,0.4848,0.4931,0.2362,0.2386
7,0.5714,0.7377,0.5714,0.5673,0.5373,0.3199,0.3331
8,0.5187,0.6937,0.5187,0.4718,0.4621,0.2435,0.2598


In [65]:
blend4_tune = tune_model(blend4)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4769,0.6311,0.4769,0.4843,0.4665,0.1873,0.1934
1,0.4484,0.6654,0.4484,0.4626,0.3873,0.1709,0.1956
2,0.4813,0.6637,0.4813,0.4774,0.4774,0.2164,0.2173
3,0.4813,0.6626,0.4813,0.4716,0.4733,0.2081,0.2097
4,0.5297,0.7036,0.5297,0.5048,0.5073,0.2733,0.2787
5,0.5231,0.6826,0.5231,0.5017,0.4971,0.2603,0.2676
6,0.5604,0.7054,0.5604,0.5363,0.5337,0.3078,0.3155
7,0.5846,0.7413,0.5846,0.5809,0.5509,0.3412,0.3549
8,0.5253,0.7000,0.5253,0.4889,0.4696,0.2538,0.2709


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [86]:
predict_model(blend4_tune);
#predict_model(blend4);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5391,0.7014,0.5391,0.4980,0.4929,0.2761,0.2899


In [88]:
newpred = predict_model(blend4_tune, data=data_unseen)
#newpred = predict_model(blend4, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5543,0.6837,0.5543,0.5096,0.5167,0.2665,0.2777


In [66]:
blend5 = blend_models([cat_tune, lgbm, et])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4835,0.6257,0.4835,0.4829,0.4756,0.1931,0.1966
1,0.4571,0.6570,0.4571,0.4662,0.4183,0.1845,0.2027
2,0.4901,0.6646,0.4901,0.4885,0.4892,0.2303,0.2304
3,0.4967,0.6626,0.4967,0.4815,0.4857,0.2303,0.2320
4,0.5055,0.6976,0.5055,0.4860,0.4902,0.2391,0.2419
5,0.5011,0.6783,0.5011,0.4751,0.4731,0.2257,0.2325
6,0.5560,0.7150,0.5560,0.5320,0.5199,0.2962,0.3075
7,0.5758,0.7361,0.5758,0.5498,0.5497,0.3330,0.3405
8,0.5451,0.7082,0.5451,0.5436,0.5016,0.2867,0.3031


In [67]:
blend5_tune = tune_model(blend5)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4725,0.6277,0.4725,0.4751,0.4659,0.1792,0.1826
1,0.4571,0.6633,0.4571,0.4649,0.4127,0.1844,0.2046
2,0.4901,0.6639,0.4901,0.4857,0.4876,0.2292,0.2293
3,0.4857,0.6656,0.4857,0.4715,0.4748,0.2132,0.2150
4,0.5209,0.7017,0.5209,0.5018,0.5045,0.2620,0.2655
5,0.5231,0.6796,0.5231,0.5027,0.4930,0.2585,0.2675
6,0.5495,0.7149,0.5495,0.5155,0.5111,0.2858,0.2967
7,0.5890,0.7383,0.5890,0.5652,0.5626,0.3534,0.3617
8,0.5341,0.7062,0.5341,0.5263,0.4833,0.2679,0.2855


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [90]:
predict_model(blend5_tune);
#predict_model(blend5);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5405,0.7024,0.5405,0.5097,0.5021,0.2802,0.2919


In [93]:
#newpred = predict_model(blend5_tune, data=data_unseen)
newpred = predict_model(blend5, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5652,0.7042,0.5652,0.5416,0.5442,0.2914,0.2980


In [68]:
blend6 = blend_models([rf, et])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4462,0.6283,0.4462,0.4468,0.4332,0.1366,0.1412
1,0.4484,0.6384,0.4484,0.4978,0.4004,0.1710,0.1946
2,0.4857,0.6411,0.4857,0.4759,0.4796,0.2208,0.2213
3,0.4879,0.6538,0.4879,0.4759,0.4751,0.2144,0.2176
4,0.5077,0.6914,0.5077,0.4815,0.4856,0.2393,0.2439
5,0.5011,0.6747,0.5011,0.4742,0.4719,0.2251,0.2324
6,0.5319,0.6900,0.5319,0.4979,0.5040,0.2636,0.2696
7,0.5846,0.7226,0.5846,0.5786,0.5575,0.3437,0.3545
8,0.5209,0.6950,0.5209,0.4928,0.4784,0.2502,0.2628


In [69]:
blend6_tune = tune_model(blend6)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4440,0.6286,0.4440,0.4457,0.4325,0.1349,0.1390
1,0.4418,0.6402,0.4418,0.4762,0.3975,0.1612,0.1824
2,0.4769,0.6390,0.4769,0.4677,0.4712,0.2078,0.2084
3,0.4923,0.6535,0.4923,0.4797,0.4791,0.2209,0.2242
4,0.5077,0.6916,0.5077,0.4856,0.4897,0.2411,0.2447
5,0.5077,0.6748,0.5077,0.4858,0.4821,0.2364,0.2431
6,0.5319,0.6920,0.5319,0.4953,0.5026,0.2632,0.2695
7,0.5802,0.7222,0.5802,0.5708,0.5539,0.3374,0.3474
8,0.5165,0.6944,0.5165,0.4822,0.4704,0.2425,0.2557


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [94]:
#predict_model(blend6_tune);
predict_model(blend6);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5372,0.6870,0.5372,0.5064,0.4941,0.2734,0.2867


In [95]:
#newpred = predict_model(blend6_tune, data=data_unseen)
newpred = predict_model(blend6, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5580,0.6849,0.5580,0.5245,0.5273,0.2769,0.2861


In [70]:
blend7 = blend_models([lgbm, xgb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4440,0.6058,0.4440,0.4320,0.4345,0.1235,0.1247
1,0.4396,0.6349,0.4396,0.4336,0.4116,0.1582,0.1674
2,0.4549,0.6436,0.4549,0.4523,0.4535,0.1766,0.1766
3,0.4681,0.6433,0.4681,0.4508,0.4561,0.1864,0.1879
4,0.4659,0.6692,0.4659,0.4568,0.4605,0.1854,0.1858
5,0.4813,0.6684,0.4813,0.4676,0.4690,0.2021,0.2044
6,0.5253,0.7031,0.5253,0.4907,0.5007,0.2560,0.2604
7,0.5275,0.7114,0.5275,0.5081,0.5152,0.2671,0.2686
8,0.5363,0.7048,0.5363,0.5244,0.5044,0.2766,0.2876


In [71]:
blend7_tune = tune_model(blend7)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4571,0.6084,0.4571,0.4450,0.4461,0.1432,0.1452
1,0.4484,0.6347,0.4484,0.4499,0.4209,0.1714,0.1820
2,0.4791,0.6512,0.4791,0.4770,0.4780,0.2133,0.2134
3,0.4725,0.6454,0.4725,0.4574,0.4618,0.1933,0.1948
4,0.4769,0.6735,0.4769,0.4662,0.4704,0.2014,0.2019
5,0.4813,0.6684,0.4813,0.4664,0.4678,0.2016,0.2041
6,0.5319,0.7052,0.5319,0.5027,0.5099,0.2670,0.2713
7,0.5275,0.7165,0.5275,0.5093,0.5160,0.2674,0.2689
8,0.5341,0.7066,0.5341,0.5231,0.5041,0.2737,0.2840


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [97]:
predict_model(blend7_tune);
#predict_model(blend7);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5200,0.6888,0.5200,0.4929,0.4996,0.2568,0.2607


In [99]:
#newpred = predict_model(blend7_tune, data=data_unseen)
newpred = predict_model(blend7, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5399,0.6842,0.5399,0.5231,0.5290,0.2626,0.2642


In [72]:
blend8 = blend_models([cat_tune, rf, lgbm, et, xgb, nb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4637,0.6245,0.4637,0.4554,0.4515,0.1577,0.1611
1,0.4462,0.6536,0.4462,0.4448,0.4009,0.1679,0.1857
2,0.4813,0.6597,0.4813,0.4757,0.4774,0.2155,0.2160
3,0.4857,0.6580,0.4857,0.4744,0.4767,0.2137,0.2153
4,0.4989,0.6945,0.4989,0.4771,0.4822,0.2286,0.2316
5,0.5297,0.6822,0.5297,0.5145,0.5067,0.2710,0.2782
6,0.5385,0.7044,0.5385,0.5052,0.5091,0.2729,0.2799
7,0.5560,0.7321,0.5560,0.5359,0.5308,0.3009,0.3084
8,0.5253,0.7097,0.5253,0.4968,0.4798,0.2563,0.2702


In [73]:
blend8_tune = tune_model(blend8)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4703,0.6264,0.4703,0.4696,0.4584,0.1721,0.1770
1,0.4505,0.6552,0.4505,0.4589,0.4073,0.1745,0.1943
2,0.4989,0.6669,0.4989,0.4952,0.4956,0.2424,0.2432
3,0.4879,0.6565,0.4879,0.4800,0.4815,0.2194,0.2207
4,0.5099,0.6978,0.5099,0.4919,0.4969,0.2476,0.2497
5,0.5231,0.6837,0.5231,0.5011,0.5009,0.2623,0.2680
6,0.5297,0.7029,0.5297,0.4981,0.5036,0.2608,0.2664
7,0.5714,0.7353,0.5714,0.5563,0.5452,0.3242,0.3331
8,0.5341,0.7063,0.5341,0.5162,0.4839,0.2685,0.2853


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [101]:
#predict_model(blend8_tune);
predict_model(blend8);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5465,0.7000,0.5465,0.5148,0.5102,0.2907,0.3015


In [103]:
newpred = predict_model(blend8_tune, data=data_unseen)
#newpred = predict_model(blend8, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5652,0.6816,0.5652,0.5398,0.5407,0.2853,0.2942


In [74]:
blend9 = blend_models([cat_tune, xgb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4484,0.6139,0.4484,0.4336,0.4383,0.1292,0.1302
1,0.4505,0.6535,0.4505,0.4462,0.4200,0.1746,0.1857
2,0.4484,0.6468,0.4484,0.4452,0.4466,0.1664,0.1665
3,0.4725,0.6558,0.4725,0.4586,0.4631,0.1943,0.1955
4,0.4835,0.6828,0.4835,0.4673,0.4723,0.2079,0.2094
5,0.5143,0.6752,0.5143,0.4994,0.4997,0.2518,0.2552
6,0.5407,0.7041,0.5407,0.5045,0.5133,0.2787,0.2845
7,0.5407,0.7235,0.5407,0.5191,0.5239,0.2831,0.2865
8,0.5407,0.7056,0.5407,0.5276,0.5066,0.2829,0.2948


In [75]:
blend9_tune = tune_model(blend9)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4571,0.6206,0.4571,0.4502,0.4493,0.1495,0.1512
1,0.4549,0.6646,0.4549,0.4767,0.4143,0.1810,0.1989
2,0.4681,0.6609,0.4681,0.4637,0.4654,0.1959,0.1961
3,0.4923,0.6658,0.4923,0.4785,0.4823,0.2241,0.2256
4,0.5011,0.6973,0.5011,0.4777,0.4830,0.2312,0.2345
5,0.5121,0.6789,0.5121,0.4964,0.4935,0.2461,0.2510
6,0.5407,0.7098,0.5407,0.5096,0.5152,0.2787,0.2844
7,0.5714,0.7362,0.5714,0.5441,0.5465,0.3270,0.3337
8,0.5341,0.7051,0.5341,0.5097,0.4868,0.2695,0.2849


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [105]:
predict_model(blend9_tune);
#predict_model(blend9);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5405,0.7017,0.5405,0.5132,0.5117,0.2838,0.2917


In [107]:
newpred = predict_model(blend9_tune, data=data_unseen)
#newpred = predict_model(blend9, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5688,0.6995,0.5688,0.5468,0.5516,0.3024,0.3068


In [76]:
blend10 = blend_models([cat_tune, lgbm, xgb])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4527,0.6137,0.4527,0.4416,0.4434,0.1397,0.1411
1,0.4505,0.6482,0.4505,0.4482,0.4193,0.1746,0.1865
2,0.4659,0.6542,0.4659,0.4630,0.4644,0.1932,0.1932
3,0.4813,0.6542,0.4813,0.4652,0.4702,0.2071,0.2085
4,0.4923,0.6835,0.4923,0.4751,0.4800,0.2206,0.2225
5,0.4945,0.6747,0.4945,0.4773,0.4771,0.2197,0.2236
6,0.5319,0.7091,0.5319,0.4991,0.5066,0.2653,0.2704
7,0.5429,0.7254,0.5429,0.5160,0.5239,0.2866,0.2900
8,0.5407,0.7086,0.5407,0.5263,0.5027,0.2819,0.2951


In [77]:
blend10_tune = tune_model(blend10)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.4835,0.6206,0.4835,0.4789,0.4755,0.1903,0.1930
1,0.4571,0.6565,0.4571,0.4521,0.4157,0.1844,0.2009
2,0.4725,0.6665,0.4725,0.4697,0.4710,0.2032,0.2032
3,0.4945,0.6617,0.4945,0.4809,0.4853,0.2281,0.2294
4,0.4967,0.6946,0.4967,0.4780,0.4827,0.2263,0.2286
5,0.4923,0.6764,0.4923,0.4694,0.4704,0.2147,0.2194
6,0.5385,0.7138,0.5385,0.5048,0.5067,0.2715,0.2794
7,0.5604,0.7363,0.5604,0.5302,0.5367,0.3111,0.3164
8,0.5451,0.7082,0.5451,0.5354,0.5021,0.2872,0.3028


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [109]:
predict_model(blend10_tune);
#predict_model(blend10);

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5381,0.7017,0.5381,0.5080,0.5094,0.2806,0.2881


In [111]:
newpred = predict_model(blend10_tune, data=data_unseen)
#newpred = predict_model(blend10, data=data_unseen)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.5471,0.7056,0.5471,0.5269,0.5316,0.2663,0.2701


In [ ]:
save_model(blend10_tune, "LigaMX_model_v3")